# Design Patterns 09: Retry & Backoff

Implementing a reusable `@retry` decorator.

In [ ]:
import time
import functools

def retry(max_attempts=3, delay=1, backoff=2):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            current_delay = delay
            
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    attempts += 1
                    print(f"Attempt {attempts} failed: {e}")
                    if attempts == max_attempts:
                        print("Max retries exhausted.")
                        raise
                    
                    print(f"Waiting {current_delay}s before retry...")
                    time.sleep(current_delay)
                    current_delay *= backoff # Exponential Backoff
        return wrapper
    return decorator

@retry(max_attempts=3, delay=0.5)
def connect_to_db():
    print("Connecting...")
    raise ConnectionError("DB Unavailable")

try:
    connect_to_db()
except ConnectionError:
    pass